# ACO-Enhanced Genetic Algorithm — Protein pH Optimizer
### Integrations: FASTA wild-type · Pre-trained model (Bootstrap) · PSSM matrix · HotSpot Wizard

| Integration | How it works |
|---|---|
| **FASTA** | Wild-type sequence is loaded directly from your `.fasta` file |
| **Pre-trained model** | Loads a saved model from the Bootstrap augmentation run (no retraining) |
| **Model choice** | User sets `MODEL_NAME` to any of the 8 trained regressors |
| **PSSM** | Per-position, per-AA scores are normalised and blended into ACO heuristic η (controlled by `PSSM_WEIGHT`). Higher PSSM score → higher sampling probability at that position |
| **HSW mutability** | Per-position mutability (0–9 scale) scales the per-position mutation rate. Low-mutability positions are mutated less aggressively |
| **HSW coevolution** | Position pairs with high z-scores are used in directed crossover — mutating one position co-mutates its strongest HSW-correlated partner |
| **Hamming distance** | Every offspring is penalised if its Hamming distance from the WT exceeds `MAX_HAMMING`. Sequences beyond the hard cap are repaired by randomly reverting excess mutations back to WT residues |

---
## 1 · Install / import dependencies

In [1]:
import subprocess, sys
for pkg in ['tqdm', 'openpyxl', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'xlrd', 'joblib']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependencies ready.')

Dependencies ready.


In [2]:
import pandas as pd
import numpy as np
import random, time, warnings, json, os
from itertools import product as iproduct
from collections import Counter, defaultdict
from pathlib import Path
warnings.filterwarnings('ignore')

import joblib

from sklearn.metrics import r2_score

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

from tqdm.notebook import tqdm
print('Imports OK.')

Imports OK.


---
## 2 · ⚙️ User configuration — **edit paths and hyperparameters here**

In [36]:
# ════════════════════════════════════════════════════════════
#  FILE PATHS
# ════════════════════════════════════════════════════════════
FASTA_FILE    = 'Wt_Afecal_Nit.fasta'                   # single-record FASTA
PSSM_FILE     = 'nitrilase_pssm_arylaceto_only.xlsx'     # rows = AA, columns = positions (1-indexed)
HSW_FILE      = 'correlated_mutations.csv'               # HotSpot Wizard coevolution CSV

# ════════════════════════════════════════════════════════════
#  PRE-TRAINED MODEL SELECTION
#  Folder: saved_models/Bootstrap__<MODEL_NAME>/
#  Available MODEL_NAME values (must match exactly):
#    'Linear_Regression'
#    'Ridge_Regression'
#    'Lasso_Regression'
#    'SVR_Linear'         (was SVR (Linear))
#    'SVR_RBF'            (was SVR (RBF))
#    'SVR_Polynomial'     (was SVR (Polynomial))
#    'Random_Forest'
#    'Gradient_Boosting'
# ════════════════════════════════════════════════════════════
MODEL_NAME    = 'Gradient_Boosting'      # ← CHANGE THIS to your preferred model
SAVED_MODELS_DIR = 'saved_models'

# ════════════════════════════════════════════════════════════
#  GA HYPERPARAMETERS
# ════════════════════════════════════════════════════════════
GA_GOAL             = 'maximize'   # 'maximize' or 'minimize'
POP_SIZE            = 150
N_GENERATIONS       = 80
CROSSOVER_RATE      = 0.75
EARLY_STOP_GENS     = 15
SEED                = 42

MUTATION_RATE_START = 0.05
MUTATION_RATE_END   = 0.01
N_MUTATIONS_MIN     = 1
N_MUTATIONS_MAX     = 3

ELITISM_FRAC_START  = 0.05
ELITISM_FRAC_END    = 0.12
TOURNAMENT_K_START  = 3
TOURNAMENT_K_END    = 8

T_START             = 1.0
T_END               = 0.10
ANNEAL_SCHEDULE     = 'cosine'     # 'linear' or 'cosine'

# ════════════════════════════════════════════════════════════
#  ACO PHEROMONE
# ════════════════════════════════════════════════════════════
PHE_INIT            = 0.1
PHE_EVAP            = 0.15
PHE_DEPOSIT_GOOD    = 1.0
PHE_DEPOSIT_BAD     = -0.5
PHE_MIN             = 0.01
PHE_MAX             = 5.0
PHE_WEIGHT_START    = 0.2
PHE_WEIGHT_END      = 0.7

# ════════════════════════════════════════════════════════════
#  MUTATION MEMORY / TABU
# ════════════════════════════════════════════════════════════
TABU_THRESHOLD      = -0.10
DIRECTED_CROSS_PROB = 0.4
BANKED_MIN_DELTA    = 0.05

# ════════════════════════════════════════════════════════════
#  PSSM INTEGRATION
# ════════════════════════════════════════════════════════════
PSSM_WEIGHT         = 0.5

# ════════════════════════════════════════════════════════════
#  HOTSPOT WIZARD INTEGRATION
# ════════════════════════════════════════════════════════════
HSW_MUTABILITY_THRESHOLD = 6
HSW_ZSCORE_COLS = ['MI z-score', 'aMIc z-score', 'DCA z-score',
                   'OMES z-score', 'SCA z-score', 'McBASC z-score', 'ELSC z-score']
HSW_COEV_ZTHRESHOLD = 0.5

# ════════════════════════════════════════════════════════════
#  HAMMING DISTANCE CONSTRAINT
# ════════════════════════════════════════════════════════════
MAX_HAMMING         = 10

print('Configuration loaded.')
print(f'  Model to use : Bootstrap__{MODEL_NAME}')

Configuration loaded.
  Model to use : Bootstrap__SVR_Linear


---
## 3 · Amino acid constants

In [28]:
MOL_WEIGHTS = {
    'A':89.1,'R':174.2,'N':132.1,'D':133.1,'C':121.2,'Q':146.2,'E':147.1,
    'G':75.1,'H':155.2,'I':131.2,'L':131.2,'K':146.2,'M':149.2,'F':165.2,
    'P':115.1,'S':105.1,'T':119.1,'W':204.2,'Y':181.2,'V':117.1
}
HYDROPATHY = {
    'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,
    'G':-0.4,'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,
    'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2
}
PKA_VALUES   = {'D':3.9,'E':4.3,'C':8.3,'Y':10.1,'H':6.0,'K':10.5,'R':12.5}
ACIDIC       = set('DE');       BASIC        = set('RHK')
HYDROPHOBIC  = set('AILMFWYV'); HYDROPHILIC  = set('STNQDE')
AROMATIC     = set('FWY');      ALIPHATIC    = set('AVILM')
POLAR        = set('STNQ');     TINY         = set('AGST')
SMALL        = set('AGSTP');    CHARGED_POS  = set('KRH')
CHARGED_NEG  = set('DE');       NONPOLAR     = set('ACFILMPVWY')
AA_STD       = 'ACDEFGHIKLMNPQRSTVWY'
AA_LIST      = list(AA_STD)
ALL_DIPEPTIDES = [''.join(p) for p in iproduct(AA_STD, repeat=2)]
ALKALINE_AA  = {'K':1.5,'R':2.0,'H':1.0}
ACIDIC_AA    = {'D':-1.5,'E':-1.2}
print(f'{len(AA_LIST)} standard AAs | {len(ALL_DIPEPTIDES)} dipeptides')

20 standard AAs | 400 dipeptides


---
## 4 · Load FASTA

In [29]:
def load_fasta(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'FASTA not found: {path}')
    header, parts = None, []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line: continue
            if line.startswith('>'):
                if header is not None: break
                header = line[1:]
            else:
                parts.append(line.upper())
    if header is None:
        raise ValueError('No FASTA header line found.')
    seq = ''.join(parts)
    bad = set(seq) - set(AA_STD)
    if bad:
        print(f'  ⚠  Non-standard characters removed: {bad}')
        seq = ''.join(c for c in seq if c in AA_STD)
    if not seq:
        raise ValueError('FASTA sequence is empty after filtering.')
    return header, seq

wt_header, wt_seq = load_fasta(FASTA_FILE)
print(f'✓ FASTA loaded')
print(f'  Header : {wt_header}')
print(f'  Length : {len(wt_seq)} aa')
print(f'  Seq    : {wt_seq[:80]}{"..." if len(wt_seq)>80 else ""}')

✓ FASTA loaded
  Header : Afecal Nit
  Length : 356 aa
  Seq    : MQTRKIVRAAAVQAASPNYDLAAGVDKTIELARQARDEGCDLIVFGETWLPGYPFHVWLGAPAWSLKYSARYYANSLSLD...


---
## 5 · Load PSSM

**What file to provide:**  
A CSV or Excel file where:
- **Rows** = the 20 standard amino acids (single-letter code, e.g. `A`, `C`, `D` …)
- **Columns** = sequence positions, **1-indexed** (column header `1` = first residue of your WT)
- **Values** = raw PSSM scores (log-odds or substitution scores; any numeric range is fine — the loader min-max normalises per position to `[0, 1]` internally)

Typical sources: PSI-BLAST PSSM export, HHblits, or a custom substitution matrix aligned to your WT.  
Columns that don't map to a valid 1-indexed position are silently ignored.

In [30]:
def load_pssm(path, seq_len):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'PSSM file not found: {path}')
    raw = (pd.read_excel(path, index_col=0)
           if path.suffix.lower() in ('.xlsx','.xls')
           else pd.read_csv(path, index_col=0))
    raw.index = [str(i).strip().upper()[:1] for i in raw.index]
    missing = [aa for aa in AA_LIST if aa not in raw.index]
    if missing:
        print(f'  ⚠  PSSM missing rows for: {missing} — defaulting to 0')
    col_pos = {}
    for c in raw.columns:
        try: col_pos[c] = int(str(c).strip()) - 1
        except ValueError: pass
    pssm = np.zeros((seq_len, len(AA_LIST)), dtype=np.float32)
    used = 0
    for col, pos in col_pos.items():
        if not (0 <= pos < seq_len): continue
        used += 1
        for ai, aa in enumerate(AA_LIST):
            if aa in raw.index:
                v = raw.loc[aa, col]
                pssm[pos, ai] = float(v) if pd.notna(v) else 0.0
    for pos in range(seq_len):
        mn, mx = pssm[pos].min(), pssm[pos].max()
        pssm[pos] = (pssm[pos]-mn)/(mx-mn) if mx > mn else np.full(len(AA_LIST), 0.5)
    print(f'  PSSM: {used} position columns matched out of {len(col_pos)} in file')
    return pssm

pssm_matrix = load_pssm(PSSM_FILE, len(wt_seq))
print(f'✓ PSSM loaded — shape {pssm_matrix.shape}  (positions × AAs)')
print(f'  Each column is min-max normalised to [0,1] per position.')
print(f'  Higher score = more evolutionarily preferred substitution at that position.')

  PSSM: 356 position columns matched out of 362 in file
✓ PSSM loaded — shape (356, 20)  (positions × AAs)
  Each column is min-max normalised to [0,1] per position.
  Higher score = more evolutionarily preferred substitution at that position.


---
## 6 · Load HotSpot Wizard  

**What file to provide (`HSW_FILE`):**  
The **Correlated mutations / Coevolution** export from HotSpot Wizard — a CSV or Excel file with one **pair of residues per row**.  
Required columns (exact names as exported by HotSpot Wizard):

| Column | Description |
|---|---|
| `Position 1` | 1-indexed residue number of the first partner |
| `Mutability 1` | HotSpot Wizard mutability score (0–9) for Position 1 |
| `Position 2` | 1-indexed residue number of the second partner |
| `Mutability 2` | HotSpot Wizard mutability score (0–9) for Position 2 |
| `MI z-score`, `aMIc z-score`, `DCA z-score`, `OMES z-score`, `SCA z-score`, `McBASC z-score`, `ELSC z-score` | Coevolution z-scores (at least one must be present; missing columns are skipped) |

How the file is used:
- **Mutability** columns → per-position mutation rate scaling (low mutability = fewer mutations attempted)
- **Z-score** columns → mean z-score per pair; pairs above `HSW_COEV_ZTHRESHOLD` drive HSW-guided crossover co-mutations

In [31]:
def load_hsw(path, seq_len):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'HotSpot Wizard file not found: {path}')
    df = (pd.read_excel(path)
          if path.suffix.lower() in ('.xlsx','.xls')
          else pd.read_csv(path))
    df.columns = [str(c).strip() for c in df.columns]

    # ── mutability ────────────────────────────────────────────
    mut_best = np.full(seq_len, np.nan, dtype=np.float32)
    for col_pos, col_mut in [('Position 1','Mutability 1'), ('Position 2','Mutability 2')]:
        if col_pos not in df.columns or col_mut not in df.columns: continue
        for _, row in df.iterrows():
            try:
                pos  = int(row[col_pos]) - 1
                mutb = float(row[col_mut])
            except (ValueError, TypeError): continue
            if 0 <= pos < seq_len:
                if np.isnan(mut_best[pos]) or mutb > mut_best[pos]:
                    mut_best[pos] = mutb
    mutability = np.where(np.isnan(mut_best), 1.0, mut_best / 9.0).astype(np.float32)
    mutability = np.clip(mutability, 0.0, 1.0)

    # ── coevolution pairs ─────────────────────────────────────
    available_z = [c for c in HSW_ZSCORE_COLS if c in df.columns]
    coev_pairs  = []
    for _, row in df.iterrows():
        try:
            p1 = int(row['Position 1']) - 1
            p2 = int(row['Position 2']) - 1
        except (ValueError, TypeError, KeyError): continue
        if not (0 <= p1 < seq_len and 0 <= p2 < seq_len): continue
        z_vals = []
        for zc in available_z:
            try:
                v = float(row.get(zc, np.nan))
                if np.isfinite(v): z_vals.append(v)
            except (ValueError, TypeError): pass
        strength = float(np.mean(z_vals)) if z_vals else 0.0
        if strength >= HSW_COEV_ZTHRESHOLD:
            coev_pairs.append((p1, p2, strength))
    coev_pairs.sort(key=lambda x: -x[2])

    n_data = (~np.isnan(mut_best)).sum()
    print(f'  HSW: {n_data} positions with mutability data | '
          f'{len(coev_pairs)} coevolving pairs (z ≥ {HSW_COEV_ZTHRESHOLD})')
    return mutability, coev_pairs

hsw_mutability, hsw_coev_pairs = load_hsw(HSW_FILE, len(wt_seq))
print(f'✓ HotSpot Wizard loaded')

# Build pos → sorted partner list for fast lookup in crossover
coev_map = defaultdict(list)
for p1, p2, strength in hsw_coev_pairs:
    coev_map[p1].append((p2, strength))
    coev_map[p2].append((p1, strength))
for pos in coev_map:
    coev_map[pos].sort(key=lambda x: -x[1])
print(f'  Coevolution map: {len(coev_map)} positions with at least one partner')

# ── Hotspot position summary ───────────────────────────────────
thr_norm = HSW_MUTABILITY_THRESHOLD / 9.0
hotspot_positions = [i for i in range(len(wt_seq)) if hsw_mutability[i] >= thr_norm]
restricted_positions = [i for i in range(len(wt_seq)) if hsw_mutability[i] < thr_norm
                        and hsw_mutability[i] > 0.0]
frozen_positions = [i for i in range(len(wt_seq)) if hsw_mutability[i] == 0.0]

print(f'\n── Hotspot / mutability breakdown (threshold = {HSW_MUTABILITY_THRESHOLD}/9) ──')
print(f'  Total residues       : {len(wt_seq)}')
print(f'  Hotspot positions    : {len(hotspot_positions):4d}  '
      f'(mutability ≥ {HSW_MUTABILITY_THRESHOLD}/9 → FULL mutation rate)')
print(f'  Restricted positions : {len(restricted_positions):4d}  '
      f'(mutability 0–{HSW_MUTABILITY_THRESHOLD-1}/9 → SCALED DOWN mutation rate)')
print(f'  Frozen positions     : {len(frozen_positions):4d}  '
      f'(mutability = 0 → effectively zero mutation rate)')
print(f'  No-HSW-data positions: {len(wt_seq) - len(hotspot_positions) - len(restricted_positions) - len(frozen_positions):4d}  '
      f'(absent from file → default full rate)')
print(f'\n  First 20 hotspot positions (1-indexed): '
      f'{[p+1 for p in hotspot_positions[:20]]}{"..." if len(hotspot_positions)>20 else ""}')

  HSW: 356 positions with mutability data | 10209 coevolving pairs (z ≥ 0.5)
✓ HotSpot Wizard loaded
  Coevolution map: 346 positions with at least one partner

── Hotspot / mutability breakdown (threshold = 6/9) ──
  Total residues       : 356
  Hotspot positions    :  142  (mutability ≥ 6/9 → FULL mutation rate)
  Restricted positions :  214  (mutability 0–5/9 → SCALED DOWN mutation rate)
  Frozen positions     :    0  (mutability = 0 → effectively zero mutation rate)
  No-HSW-data positions:    0  (absent from file → default full rate)

  First 20 hotspot positions (1-indexed): [5, 6, 22, 28, 29, 30, 33, 37, 38, 41, 55, 57, 59, 60, 61, 62, 63, 64, 65, 66]...


---
## 7 · Load pre-trained model from saved_models/

No retraining happens here. The scaler, PCA, and model were all saved by the ML pipeline script.  
The `feature_names.json` file tells the predictor which feature columns to use and in what order.

In [37]:
run_dir = Path(SAVED_MODELS_DIR) / f'Bootstrap__{MODEL_NAME}'

if not run_dir.exists():
    available = sorted(str(p.name) for p in Path(SAVED_MODELS_DIR).glob('Bootstrap__*'))
    raise FileNotFoundError(
        f'Model directory not found: {run_dir}\n'
        f'Available Bootstrap models:\n' +
        '\n'.join(f'  {a}' for a in available)
    )

model        = joblib.load(run_dir / 'model.pkl')
pca          = joblib.load(run_dir / 'pca.pkl')
scaler       = joblib.load(run_dir / 'scaler.pkl')
feature_names = json.load(open(run_dir / 'feature_names.json'))
metadata     = json.load(open(run_dir / 'metadata.json'))

print(f'✓ Model loaded from: {run_dir}')
print(f'  Augmentation   : {metadata["augmentation"]}')
print(f'  Model type     : {metadata["model"]}')
print(f'  Training R²    : (Bootstrap-augmented dataset)')
print(f'  Test R²        : {metadata["test_r2"]:.4f}')
print(f'  RMSE           : {metadata["rmse"]:.4f}')
print(f'  CV R²          : {metadata["cv_r2_mean"]:.4f} ± {metadata["cv_r2_std"]:.4f}')
print(f'  Dataset size   : {metadata["dataset_size"]} (augmented)')
print(f'  PCA components : {metadata["pca_components"]}')
print(f'  Feature count  : {len(feature_names)}')

✓ Model loaded from: saved_models\Bootstrap__SVR_Linear
  Augmentation   : Bootstrap
  Model type     : SVR (Linear)
  Training R²    : (Bootstrap-augmented dataset)
  Test R²        : 0.7068
  RMSE           : 0.5287
  CV R²          : -623.8325 ± 1203.0239
  Dataset size   : 240 (augmented)
  PCA components : 36
  Feature count  : 25948


---
## 8 · Feature extraction functions

In [38]:
def extract_biochemical_features(sequences, silent=False):
    seqs = [str(s).upper() for s in sequences]
    n    = len(seqs)
    feat_names = ['length','molecular_weight','estimated_pI']
    for ph in [5.0,6.0,7.0,7.4,8.0,9.0]:
        feat_names += [f'net_charge_pH_{ph}',f'charge_density_pH_{ph}']
    feat_names += ['aromaticity','GRAVY','aliphatic_index','avg_pKa','pKa_std']
    cls_names = ['acidic','basic','hydrophobic','hydrophilic','aromatic',
                 'aliphatic','polar','tiny','small','charged_pos','charged_neg','nonpolar']
    for c in cls_names:
        feat_names += [f'num_{c}',f'ratio_{c}']
    feat_names += ['charge_to_polar_ratio','hydrophobic_to_hydrophilic_ratio',
                   'aromatic_to_aliphatic_ratio','pos_to_neg_charge_ratio','polar_to_nonpolar_ratio']
    feat_names += [f'{aa}_freq' for aa in AA_STD]
    X = np.zeros((n, len(feat_names)), dtype=np.float32)
    cls_sets = {
        'acidic':ACIDIC,'basic':BASIC,'hydrophobic':HYDROPHOBIC,'hydrophilic':HYDROPHILIC,
        'aromatic':AROMATIC,'aliphatic':ALIPHATIC,'polar':POLAR,'tiny':TINY,'small':SMALL,
        'charged_pos':CHARGED_POS,'charged_neg':CHARGED_NEG,'nonpolar':NONPOLAR
    }
    it = range(n) if silent else tqdm(range(n), desc='Biochemical feats', leave=False)
    for i in it:
        seq = seqs[i]; L = len(seq)
        if L == 0: continue
        col = 0
        X[i,col]=L; col+=1
        X[i,col]=sum(MOL_WEIGHTS.get(a,0) for a in seq); col+=1
        nd=seq.count('D'); ne=seq.count('E'); nc=seq.count('C')
        ny=seq.count('Y'); nh=seq.count('H'); nk=seq.count('K'); nr=seq.count('R')
        X[i,col]=7.0+(nh*0.5+nk+nr-nd-ne)*0.5; col+=1
        for ph in [5.0,6.0,7.0,7.4,8.0,9.0]:
            charge = 0
            if ph<10.5: charge+=nk
            if ph<12.5: charge+=nr
            if ph<6.0:  charge+=nh
            if ph>3.9:  charge-=nd
            if ph>4.3:  charge-=ne
            if ph>8.3:  charge-=nc
            if ph>10.1: charge-=ny
            X[i,col]=charge; col+=1
            X[i,col]=charge/L; col+=1
        ar=sum(1 for a in seq if a in AROMATIC)
        X[i,col]=ar/L; col+=1
        X[i,col]=sum(HYDROPATHY.get(a,0) for a in seq)/L; col+=1
        X[i,col]=100*(seq.count('A')/L+2.9*seq.count('V')/L+3.9*(seq.count('I')+seq.count('L'))/L); col+=1
        pka_list=[PKA_VALUES[a] for a in seq if a in PKA_VALUES]
        X[i,col]=np.mean(pka_list) if pka_list else 7.0; col+=1
        X[i,col]=np.std(pka_list) if len(pka_list)>1 else 0; col+=1
        cnts={c:sum(1 for a in seq if a in s) for c,s in cls_sets.items()}
        cnts['aromatic']=ar
        for c in cls_names:
            X[i,col]=cnts[c]; col+=1
            X[i,col]=cnts[c]/L; col+=1
        X[i,col]=(cnts['acidic']+cnts['basic'])/(cnts['polar']+1); col+=1
        X[i,col]=cnts['hydrophobic']/(cnts['hydrophilic']+1); col+=1
        X[i,col]=cnts['aromatic']/(cnts['aliphatic']+1); col+=1
        X[i,col]=cnts['charged_pos']/(cnts['charged_neg']+1); col+=1
        X[i,col]=cnts['polar']/(cnts['nonpolar']+1); col+=1
        for aa in AA_STD:
            X[i,col]=seq.count(aa)/L; col+=1
    return pd.DataFrame(X, columns=feat_names)


def extract_dipeptide_features(sequences, silent=False):
    seqs    = [str(s).upper() for s in sequences]
    col_idx = {dp:i for i,dp in enumerate(ALL_DIPEPTIDES)}
    X       = np.zeros((len(seqs),400), dtype=np.float32)
    it = range(len(seqs)) if silent else tqdm(range(len(seqs)), desc='Dipeptide feats', leave=False)
    for i in it:
        seq=seqs[i]; L=len(seq)
        if L<2: continue
        total=L-1
        for k in range(total):
            dp=seq[k:k+2]
            if dp in col_idx: X[i,col_idx[dp]]+=1
        X[i]/=total
    return pd.DataFrame(X, columns=[f'dipep_{d}' for d in ALL_DIPEPTIDES])


def build_kmer_vocab(sequences, k, min_count=2):
    counter=Counter()
    for seq in sequences:
        seq=str(seq).upper()
        counter.update(set(seq[i:i+k] for i in range(len(seq)-k+1)))
    return sorted(km for km,cnt in counter.items() if cnt>=min_count)


def extract_kmer_features(sequences, vocab, k, prefix, silent=False):
    seqs    = [str(s).upper() for s in sequences]
    col_idx = {km:i for i,km in enumerate(vocab)}
    X       = np.zeros((len(seqs),len(vocab)), dtype=np.float32)
    it = range(len(seqs)) if silent else tqdm(range(len(seqs)), desc=f'{prefix} feats', leave=False)
    for i in it:
        seq=seqs[i]; L=len(seq)
        if L<k: continue
        total=L-k+1
        for j in range(total):
            km=seq[j:j+k]
            if km in col_idx: X[i,col_idx[km]]+=1
        X[i]/=total
    return pd.DataFrame(X, columns=[f'{prefix}_{km}' for km in vocab])


def extract_features_for_ml(sequences, tri_vocab=None, silent=False):
    bio_df = extract_biochemical_features(sequences, silent=silent)
    di_df  = extract_dipeptide_features(sequences, silent=silent)
    if tri_vocab is None:
        tri_vocab = build_kmer_vocab(sequences, k=3, min_count=2)
    tri_df = extract_kmer_features(sequences, tri_vocab, k=3, prefix='tripep', silent=silent)
    return pd.concat([bio_df, di_df, tri_df], axis=1).fillna(0), tri_vocab


print('Feature extraction functions defined.')

Feature extraction functions defined.


---
## 9 · Build predictor and confirm on WT  

The `BatchPredictor` aligns the feature matrix to the **exact column order** saved in `feature_names.json`, then applies scaler → PCA → model. Missing columns are zero-filled.

In [39]:
# ── Derive tri_vocab from feature_names saved with the model ──────
# This ensures the predictor uses the SAME tripeptide vocabulary as training.
tri_vocab = sorted(set(
    fn[len('tripep_'):] for fn in feature_names if fn.startswith('tripep_')
))
print(f'  Tripeptide vocab size (from saved model): {len(tri_vocab)}')


class BatchPredictor:
    def __init__(self, feature_names, tri_vocab, scaler, pca, model):
        self.feature_names = feature_names
        self.tri_vocab     = tri_vocab
        self.scaler        = scaler
        self.pca           = pca
        self.model         = model

    def predict_batch(self, sequences):
        X_df, _ = extract_features_for_ml(sequences, tri_vocab=self.tri_vocab, silent=True)
        for col in self.feature_names:
            if col not in X_df.columns:
                X_df[col] = 0.0
        X_arr = X_df[self.feature_names].fillna(0).values.astype(np.float32)
        X_sc  = self.scaler.transform(X_arr)
        X_pc  = self.pca.transform(X_sc)
        return self.model.predict(X_pc).astype(float)

    def predict_one(self, seq):
        return float(self.predict_batch([seq])[0])


predictor = BatchPredictor(feature_names, tri_vocab, scaler, pca, model)
wt_pred   = predictor.predict_one(wt_seq)

print(f'\n✓ Predictor ready')
print(f'  Wild-type predicted pH : {wt_pred:.4f}')
print(f'  Model                  : {MODEL_NAME}  (Bootstrap augmentation)')

  Tripeptide vocab size (from saved model): 4161

✓ Predictor ready
  Wild-type predicted pH : 7.2550
  Model                  : SVR_Linear  (Bootstrap augmentation)


---
## 10 · PSSM-aware ACO pheromone matrix

In [40]:
class PheromoneMatrix:
    """
    τ[pos][aa]  — pheromone: updated from GA evaluations.
    η[pos][aa]  — heuristic: WEIGHTED BLEND of
                  (1-PSSM_WEIGHT) × biochemical prior
                + PSSM_WEIGHT     × normalised PSSM score
    """
    def __init__(self, seq_len, pssm, goal='maximize'):
        self.L      = seq_len
        self.goal   = goal
        self.aa_idx = {aa: i for i, aa in enumerate(AA_LIST)}
        n_aa        = len(AA_LIST)
        self.tau = np.full((seq_len, n_aa), PHE_INIT, dtype=np.float64)
        eta_bio = np.ones((seq_len, n_aa), dtype=np.float64)
        for aa, bonus in ALKALINE_AA.items():
            if aa in self.aa_idx: eta_bio[:, self.aa_idx[aa]] += bonus
        for aa, pen in ACIDIC_AA.items():
            if aa in self.aa_idx:
                c = self.aa_idx[aa]
                eta_bio[:, c] = np.maximum(0.01, eta_bio[:, c] + pen)
        for pos in range(seq_len):
            mn, mx = eta_bio[pos].min(), eta_bio[pos].max()
            eta_bio[pos] = (eta_bio[pos]-mn)/(mx-mn) if mx>mn else np.full(n_aa, 0.5)
        pssm_use = np.full((seq_len, n_aa), 0.5, dtype=np.float64)
        rows = min(pssm.shape[0], seq_len)
        pssm_use[:rows, :] = pssm[:rows, :]
        self.eta = ((1.0 - PSSM_WEIGHT) * eta_bio + PSSM_WEIGHT * pssm_use)
        if goal == 'minimize':
            self.eta = 1.0 / (self.eta + 1e-6)
            for pos in range(seq_len):
                s = self.eta[pos].sum()
                if s > 0: self.eta[pos] /= s / n_aa

    def evaporate(self):
        self.tau = np.clip(self.tau * (1.0 - PHE_EVAP), PHE_MIN, PHE_MAX)

    def deposit(self, wt_seq, mut_seq, delta_ph):
        for pos, (w, m) in enumerate(zip(wt_seq, mut_seq)):
            if w == m: continue
            ac = self.aa_idx.get(m)
            if ac is None: continue
            if   delta_ph >  0:    dep = PHE_DEPOSIT_GOOD * abs(delta_ph)
            elif delta_ph < -0.05: dep = PHE_DEPOSIT_BAD  * abs(delta_ph)
            else:                  dep = 0.0
            self.tau[pos, ac] = np.clip(self.tau[pos, ac] + dep, PHE_MIN, PHE_MAX)

    def get_probs(self, pos, wt_aa, phe_weight):
        scores = (self.tau[pos] ** phe_weight) * (self.eta[pos] ** (1.0 - phe_weight))
        wc = self.aa_idx.get(wt_aa)
        if wc is not None: scores[wc] = 0.0
        s = scores.sum()
        if s <= 0:
            scores = np.ones(len(AA_LIST))
            if wc is not None: scores[wc] = 0.0
            s = scores.sum()
        return scores / s

    def choose_aa(self, pos, wt_aa, phe_weight, rng):
        return AA_LIST[rng.choice(len(AA_LIST), p=self.get_probs(pos, wt_aa, phe_weight))]


print('✓ PheromoneMatrix defined (PSSM-aware, PSSM_WEIGHT={})'.format(PSSM_WEIGHT))

✓ PheromoneMatrix defined (PSSM-aware, PSSM_WEIGHT=0.5)


---
## 11 · Mutation memory, temperature schedule, Individual

In [41]:
class MutationMemory:
    """Tracks best ΔpH for every (pos, wt_aa, mut_aa) triple seen."""
    def __init__(self):
        self._rec = defaultdict(lambda: -np.inf)

    def record(self, wt_seq, mut_seq, delta):
        for pos, (w, m) in enumerate(zip(wt_seq, mut_seq)):
            if w != m and delta > self._rec[(pos,w,m)]:
                self._rec[(pos,w,m)] = delta

    @property
    def banked(self):
        return {k:v for k,v in self._rec.items() if v >= BANKED_MIN_DELTA}

    @property
    def tabu_positions(self):
        pb = defaultdict(lambda: -np.inf)
        for (pos,w,m), d in self._rec.items():
            if d > pb[pos]: pb[pos] = d
        return {pos for pos,best in pb.items() if best < TABU_THRESHOLD}


def temperature(gen, n_gens):
    frac = gen / max(n_gens-1, 1)
    if ANNEAL_SCHEDULE == 'cosine':
        t = T_END + 0.5*(T_START-T_END)*(1+np.cos(np.pi*frac))
    else:
        t = T_START - frac*(T_START-T_END)
    return float(np.clip(t, T_END, T_START))


def lerp(start, end, T):
    frac = (T - T_END) / max(T_START - T_END, 1e-8)
    return end + frac*(start-end)


class Individual:
    __slots__ = ('seq','fitness')
    def __init__(self, seq, fitness=None):
        self.seq=seq; self.fitness=fitness
    def copy(self): return Individual(self.seq, self.fitness)


print('✓ MutationMemory · temperature · Individual defined.')

✓ MutationMemory · temperature · Individual defined.


---
## 12 · Population, evaluation, selection

In [42]:
def init_population(wt_seq, pop_size, rng):
    pop = [Individual(wt_seq) for _ in range(max(1, pop_size//5))]
    while len(pop) < pop_size:
        seq = list(wt_seq)
        for pos in rng.choice(len(seq), size=rng.integers(1,4), replace=False):
            cands = [aa for aa in AA_LIST if aa != seq[pos]]
            seq[pos] = cands[rng.integers(len(cands))]
        pop.append(Individual(''.join(seq)))
    return pop


def evaluate_population(pop, predictor, goal):
    uneval = [ind for ind in pop if ind.fitness is None]
    if not uneval: return
    preds = predictor.predict_batch([ind.seq for ind in uneval])
    for ind, p in zip(uneval, preds):
        ind.fitness = p if goal=='maximize' else -p


def tournament_select(pop, k, rng):
    idx  = rng.choice(len(pop), size=k, replace=False)
    best = max(idx, key=lambda i: pop[i].fitness)
    return pop[best].copy()


print('✓ Population functions defined.')

✓ Population functions defined.


---
## 13 · Crossover (banked-mutation grafting + HSW coevolution co-mutation)

In [43]:
def crossover(p1, p2, rng, memory, wt_seq, T, pheromone, phe_weight):
    if len(p1.seq) < 2:
        return p1.copy(), p2.copy()
    pt     = rng.integers(1, len(p1.seq))
    child1 = Individual(p1.seq[:pt] + p2.seq[pt:])
    child2 = Individual(p2.seq[:pt] + p1.seq[pt:])
    exploit = 1.0 - (T - T_END) / max(T_START - T_END, 1e-8)
    if rng.random() < DIRECTED_CROSS_PROB * exploit:
        banked = memory.banked
        if banked:
            pos, wt_aa, mut_aa = random.choice(list(banked.keys()))
            if pos < len(child1.seq) and child1.seq[pos] in (wt_seq[pos], wt_aa):
                seq = list(child1.seq)
                seq[pos] = mut_aa
                if pos in coev_map:
                    partner_pos = coev_map[pos][0][0]
                    if partner_pos < len(seq):
                        partner_wt  = wt_seq[partner_pos] if partner_pos < len(wt_seq) else seq[partner_pos]
                        seq[partner_pos] = pheromone.choose_aa(partner_pos, partner_wt, phe_weight, rng)
                child1 = Individual(''.join(seq))
    return child1, child2


print('✓ Crossover defined (HSW coevolution-aware).')

✓ Crossover defined (HSW coevolution-aware).


---
## 14 · ACO mutation (HSW mutability-scaled)

In [44]:
def mutate_aco(ind, wt_seq, pheromone, memory, mut_rate,
               n_min, n_max, phe_weight, hsw_mut, rng):
    seq   = list(ind.seq)
    L     = len(seq)
    tabu  = memory.tabu_positions
    thr   = HSW_MUTABILITY_THRESHOLD / 9.0
    eligible = [i for i in range(L) if i not in tabu] or list(range(L))
    to_mutate = []
    for i in eligible:
        mut_score  = float(hsw_mut[i]) if i < len(hsw_mut) else 1.0
        scale      = 1.0 if mut_score >= thr else mut_score / (thr + 1e-8)
        if rng.random() < mut_rate * scale:
            to_mutate.append(i)
    if rng.random() < 0.3 and len(to_mutate) < n_min:
        extra     = rng.choice(eligible, size=min(rng.integers(n_min, n_max+1), len(eligible)), replace=False).tolist()
        to_mutate = list(set(to_mutate + extra))
    for pos in to_mutate:
        wt_aa    = wt_seq[pos] if pos < len(wt_seq) else seq[pos]
        seq[pos] = pheromone.choose_aa(pos, wt_aa, phe_weight, rng)
    return Individual(''.join(seq))


print('✓ ACO mutation defined (HSW mutability-scaled).')

✓ ACO mutation defined (HSW mutability-scaled).


---
## 15 · Main ACO-GA loop

In [45]:
def describe_mutations(wt_seq, mut_seq):
    muts = [f'{w}{i+1}{m}' for i,(w,m) in enumerate(zip(wt_seq,mut_seq)) if w!=m]
    return ', '.join(muts) if muts else 'WT'


def hamming(seq_a, seq_b):
    return sum(a != b for a, b in zip(seq_a, seq_b))


def hamming_similarity(seq_a, seq_b):
    L = min(len(seq_a), len(seq_b))
    return (1.0 - hamming(seq_a, seq_b) / L) if L > 0 else 1.0


def repair_hamming(seq, wt_seq, max_hamming, rng):
    if max_hamming is None: return seq
    mut_pos = [i for i,(w,m) in enumerate(zip(wt_seq, seq)) if w != m]
    excess  = len(mut_pos) - max_hamming
    if excess <= 0: return seq
    to_revert = rng.choice(len(mut_pos), size=excess, replace=False)
    sl = list(seq)
    for idx in to_revert:
        sl[mut_pos[idx]] = wt_seq[mut_pos[idx]]
    return ''.join(sl)


print(f'✓ Hamming helpers defined. MAX_HAMMING = {MAX_HAMMING}')


def run_aco_ga(wt_seq, predictor, wt_pred, pssm, hsw_mutability,
               goal=GA_GOAL, pop_size=POP_SIZE, n_generations=N_GENERATIONS,
               crossover_rate=CROSSOVER_RATE, early_stop=EARLY_STOP_GENS, seed=SEED):

    rng = np.random.default_rng(seed)
    random.seed(seed)

    pheromone = PheromoneMatrix(len(wt_seq), pssm=pssm, goal=goal)
    memory    = MutationMemory()
    history, all_ever = [], []

    print(f'  Goal            : {goal} predicted pH')
    print(f'  Model           : {MODEL_NAME}  (Bootstrap augmentation)')
    print(f'  Pop / Gens      : {pop_size} / {n_generations}')
    print(f'  PSSM weight     : {PSSM_WEIGHT}')
    print(f'  HSW mut thr     : {HSW_MUTABILITY_THRESHOLD}/9')
    print(f'  Coev pairs used : {len(hsw_coev_pairs)}')
    print(f'  Anneal schedule : {ANNEAL_SCHEDULE}  {T_START}→{T_END}')
    print(f'  Max Hamming     : {MAX_HAMMING}')
    print()

    population = init_population(wt_seq, pop_size, rng)
    for ind in population:
        ind.seq = repair_hamming(ind.seq, wt_seq, MAX_HAMMING, rng)
    evaluate_population(population, predictor, goal)

    for ind in population:
        ph_raw = ind.fitness if goal=='maximize' else -ind.fitness
        delta  = ph_raw - wt_pred
        memory.record(wt_seq, ind.seq, delta)
        pheromone.deposit(wt_seq, ind.seq, delta)

    all_ever.extend([ind.copy() for ind in population])
    best_ever, stagnant = -np.inf, 0

    for gen in tqdm(range(n_generations), desc='ACO-GA generations'):
        T          = temperature(gen, n_generations)
        mut_rate   = lerp(MUTATION_RATE_START, MUTATION_RATE_END, T)
        phe_weight = lerp(PHE_WEIGHT_START, PHE_WEIGHT_END, T)
        exploit    = 1.0 - (T-T_END)/max(T_START-T_END, 1e-8)
        el_frac    = ELITISM_FRAC_START + exploit*(ELITISM_FRAC_END-ELITISM_FRAC_START)
        tourn_k    = int(round(TOURNAMENT_K_START + exploit*(TOURNAMENT_K_END-TOURNAMENT_K_START)))
        n_elite    = max(1, int(pop_size*el_frac))

        population.sort(key=lambda x: x.fitness, reverse=True)

        gen_best   = population[0].fitness
        gen_mean   = np.mean([x.fitness for x in population])
        gen_std    = np.std ([x.fitness for x in population])
        disp_b     = gen_best if goal=='maximize' else -gen_best
        disp_m     = gen_mean if goal=='maximize' else -gen_mean

        ham_vals   = [hamming(wt_seq, ind.seq) for ind in population]
        ham_best   = hamming(wt_seq, population[0].seq)
        ham_mean   = float(np.mean(ham_vals))
        ham_max    = int(np.max(ham_vals))

        history.append({
            'generation'    : gen,
            'best_pH'       : disp_b,
            'mean_pH'       : disp_m,
            'std_fitness'   : gen_std,
            'best_fitness'  : gen_best,
            'mean_fitness'  : gen_mean,
            'temperature'   : T,
            'mutation_rate' : mut_rate,
            'phe_weight'    : phe_weight,
            'tournament_k'  : tourn_k,
            'n_tabu_pos'    : len(memory.tabu_positions),
            'n_banked'      : len(memory.banked),
            'best_mutations': describe_mutations(wt_seq, population[0].seq),
            'hamming_best'  : ham_best,
            'hamming_mean'  : round(ham_mean, 2),
            'hamming_max'   : ham_max,
        })

        if gen_best > best_ever + 1e-6:
            best_ever = gen_best; stagnant = 0
        else:
            stagnant += 1
        if stagnant >= early_stop and gen >= early_stop:
            print(f'\n⏹  Early stop at gen {gen} (no improvement for {early_stop} gens)')
            break

        pheromone.evaporate()

        next_pop = [population[i].copy() for i in range(n_elite)]
        while len(next_pop) < pop_size:
            p1 = tournament_select(population, tourn_k, rng)
            p2 = tournament_select(population, tourn_k, rng)
            if rng.random() < crossover_rate:
                c1, c2 = crossover(p1, p2, rng, memory, wt_seq, T, pheromone, phe_weight)
            else:
                c1, c2 = p1.copy(), p2.copy()
            c1 = mutate_aco(c1, wt_seq, pheromone, memory, mut_rate,
                            N_MUTATIONS_MIN, N_MUTATIONS_MAX, phe_weight, hsw_mutability, rng)
            c2 = mutate_aco(c2, wt_seq, pheromone, memory, mut_rate,
                            N_MUTATIONS_MIN, N_MUTATIONS_MAX, phe_weight, hsw_mutability, rng)
            c1.seq = repair_hamming(c1.seq, wt_seq, MAX_HAMMING, rng)
            c2.seq = repair_hamming(c2.seq, wt_seq, MAX_HAMMING, rng)
            next_pop.extend([c1, c2])

        population = next_pop[:pop_size]
        evaluate_population(population, predictor, goal)

        for ind in population:
            ph_raw = ind.fitness if goal=='maximize' else -ind.fitness
            pheromone.deposit(wt_seq, ind.seq, ph_raw - wt_pred)
            memory.record(wt_seq, ind.seq, ph_raw - wt_pred)

        all_ever.extend([ind.copy() for ind in population])

    population.sort(key=lambda x: x.fitness, reverse=True)
    return population, history, all_ever, memory, pheromone


print('✓ ACO-GA loop defined (Hamming-constrained).')

✓ Hamming helpers defined. MAX_HAMMING = 10
✓ ACO-GA loop defined (Hamming-constrained).


---
## 16 · Run the optimiser

In [46]:
print('Starting ACO-GA optimisation...')
t_start = time.time()

final_pop, history, all_ever, memory, pheromone = run_aco_ga(
    wt_seq        = wt_seq,
    predictor     = predictor,
    wt_pred       = wt_pred,
    pssm          = pssm_matrix,
    hsw_mutability= hsw_mutability,
)

elapsed = time.time() - t_start
print(f'\nACO-GA finished in {elapsed:.1f}s | {len(history)} generations')

# ── Deduplicate across all generations ───────────────────────────
seen = {}
for ind in all_ever:
    if ind.seq not in seen or ind.fitness > seen[ind.seq]:
        seen[ind.seq] = ind.fitness
unique_inds = sorted([Individual(s,f) for s,f in seen.items()],
                     key=lambda x: x.fitness, reverse=True)

best      = unique_inds[0]
best_ph   = best.fitness if GA_GOAL=='maximize' else -best.fitness
best_muts = describe_mutations(wt_seq, best.seq)
n_changes = sum(1 for w,m in zip(wt_seq, best.seq) if w!=m)

print(f'\n🏆  Best sequence')
print(f'    Mutations   : {best_muts}')
print(f'    N changes   : {n_changes}')
print(f'    Hamming dist: {hamming(wt_seq, best.seq)}  (similarity: {hamming_similarity(wt_seq, best.seq):.3f})')
print(f'    Pred pH     : {best_ph:.4f}  (WT: {wt_pred:.4f} | ΔpH: {best_ph-wt_pred:+.4f})')
print(f'\n🧬  Memory')
print(f'    Banked      : {len(memory.banked)} beneficial mutations')
print(f'    Tabu        : {len(memory.tabu_positions)} positions blocked')

top50_ham  = [hamming(wt_seq, ind.seq) for ind in unique_inds[:50]]
print(f'\n📏  Hamming stats (top-50 unique sequences)')
print(f'    Top-50 mean Hamming : {np.mean(top50_ham):.2f}')
print(f'    Top-50 max  Hamming : {max(top50_ham)}')
print(f'    Top-50 min  Hamming : {min(top50_ham)}')
print(f'    MAX_HAMMING cap     : {MAX_HAMMING}')

Starting ACO-GA optimisation...
  Goal            : maximize predicted pH
  Model           : SVR_Linear  (Bootstrap augmentation)
  Pop / Gens      : 150 / 80
  PSSM weight     : 0.5
  HSW mut thr     : 6/9
  Coev pairs used : 10209
  Anneal schedule : cosine  1.0→0.1
  Max Hamming     : 10



ACO-GA generations:   0%|          | 0/80 [00:00<?, ?it/s]


ACO-GA finished in 13577.1s | 80 generations

🏆  Best sequence
    Mutations   : L31M, V44Q, S69K, F84E, A91F, D118I, T138P, V139Q, H198C, L262T
    N changes   : 10
    Hamming dist: 10  (similarity: 0.972)
    Pred pH     : 7.3201  (WT: 7.2550 | ΔpH: +0.0651)

🧬  Memory
    Banked      : 731 beneficial mutations
    Tabu        : 0 positions blocked

📏  Hamming stats (top-50 unique sequences)
    Top-50 mean Hamming : 9.96
    Top-50 max  Hamming : 10
    Top-50 min  Hamming : 9
    MAX_HAMMING cap     : 10


---
## 17 · Visualisation

In [47]:
gens  = [h['generation']  for h in history]
bphs  = [h['best_pH']     for h in history]
mphs  = [h['mean_pH']     for h in history]
temps = [h['temperature'] for h in history]
mrates= [h['mutation_rate'] for h in history]
pweights=[h['phe_weight'] for h in history]
banked=[h['n_banked']     for h in history]
tabus = [h['n_tabu_pos']  for h in history]
hbest = [h['hamming_best'] for h in history]
hmean = [h['hamming_mean'] for h in history]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle(f'ACO-GA Convergence  |  Model: {MODEL_NAME} (Bootstrap)', fontsize=13, fontweight='bold')

ax = axes[0,0]
ax.plot(gens, bphs, 'b-', lw=2, label='Best pH')
ax.plot(gens, mphs, 'g--', lw=1.2, label='Mean pH', alpha=0.7)
ax.axhline(wt_pred, color='red', lw=1, linestyle=':', label=f'WT ({wt_pred:.3f})')
ax.set_xlabel('Generation'); ax.set_ylabel('Predicted pH'); ax.set_title('pH Convergence')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[0,1]
ax.plot(gens, temps, 'r-', lw=2, label='Temperature')
ax.plot(gens, mrates, 'b--', lw=1.5, label='Mut rate')
ax.plot(gens, pweights, 'g-.', lw=1.5, label='Phe weight')
ax.set_xlabel('Generation'); ax.set_title('Annealing Schedules')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1,0]
ax.plot(gens, banked, 'purple', lw=2, label='Banked mutations')
ax2b = ax.twinx()
ax2b.plot(gens, tabus, 'orange', lw=1.5, linestyle='--', label='Tabu positions')
ax.set_xlabel('Generation'); ax.set_ylabel('Banked', color='purple')
ax2b.set_ylabel('Tabu', color='orange'); ax.set_title('Memory System')
ax.grid(True, alpha=0.3)

ax = axes[1,1]
ax.plot(gens, hbest, 'b-', lw=2, label='Best Hamming')
ax.plot(gens, hmean, 'g--', lw=1.5, label='Mean Hamming', alpha=0.7)
ax.axhline(MAX_HAMMING, color='red', lw=1, linestyle=':', label=f'Cap ({MAX_HAMMING})')
ax.set_xlabel('Generation'); ax.set_ylabel('Hamming distance'); ax.set_title('Diversity (Hamming from WT)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[2,0]
top20_ph  = [((ind.fitness if GA_GOAL=='maximize' else -ind.fitness)) for ind in unique_inds[:20]]
top20_ham = [hamming(wt_seq, ind.seq) for ind in unique_inds[:20]]
ax.scatter(top20_ham, top20_ph, c=range(20), cmap='RdYlGn', s=80, zorder=3)
ax.axhline(wt_pred, color='red', lw=1, linestyle=':', label=f'WT ({wt_pred:.3f})')
ax.set_xlabel('Hamming distance from WT'); ax.set_ylabel('Predicted pH')
ax.set_title('Top-20 Unique Sequences: pH vs Hamming')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[2,1]
mut_counter = Counter()
for ind in unique_inds[:50]:
    for i,(w,m) in enumerate(zip(wt_seq,ind.seq)):
        if w!=m: mut_counter[f'{w}{i+1}{m}']+=1
top_muts = mut_counter.most_common(15)
if top_muts:
    labels, counts = zip(*top_muts)
    ax.barh(range(len(labels)), counts, color='steelblue')
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis(); ax.set_xlabel('Frequency in top-50')
    ax.set_title('Most Frequent Mutations (top-50 sequences)')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('aco_ga_convergence.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved aco_ga_convergence.png')

✓ Saved aco_ga_convergence.png


---
## 18 · Excel results export

In [48]:
def write_excel(top_inds, history, wt_seq, wt_pred, memory, goal):
    wb  = Workbook()
    thin = Side(style='thin', color='FF888888')
    bdr  = Border(left=thin,right=thin,top=thin,bottom=thin)

    def _hdr(ws,r,c,v):
        cell=ws.cell(r,c,v)
        cell.font=Font(bold=True,color='FFFFFFFF')
        cell.fill=PatternFill('solid',fgColor='FF2E4057')
        cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
        cell.border=bdr

    def _cell(ws,r,c,v,bg='FFFFFFFF',fmt=None,align='center'):
        cell=ws.cell(r,c,v)
        cell.fill=PatternFill('solid',fgColor=bg)
        cell.alignment=Alignment(horizontal=align,vertical='center')
        cell.border=bdr
        if fmt: cell.number_format=fmt

    def _aw(ws):
        for col in ws.columns:
            ml=max((len(str(c.value)) for c in col if c.value),default=8)
            ws.column_dimensions[get_column_letter(col[0].column)].width=min(ml+2,50)

    def ph_bg(ph):
        if ph>=8.0: return 'FF4CAF50'
        if ph>=7.5: return 'FF8BC34A'
        if ph>=7.0: return 'FFFFEB3B'
        if ph>=6.5: return 'FFFF9800'
        return 'FFF44336'

    def rank_bg(i):
        if i==0: return 'FFFFD700'
        if i==1: return 'FFC0C0C0'
        if i==2: return 'FFCD7F32'
        return 'FFF5F5F5' if i%2==0 else 'FFFFFFFF'

    # ── Sheet 1: Summary ─────────────────────────────────────────
    ws1 = wb.active; ws1.title='Summary'
    summary_rows = [
        ('WT Header',           wt_header),
        ('WT Length',           f'{len(wt_seq)} aa'),
        ('WT Predicted pH',     round(wt_pred,4)),
        ('Goal',                goal),
        ('Model',               MODEL_NAME),
        ('Augmentation',        'Bootstrap'),
        ('Model Test R²',       round(metadata['test_r2'],4)),
        ('Model RMSE',          round(metadata['rmse'],4)),
        ('PCA Components',      metadata['pca_components']),
        ('PSSM Weight',         PSSM_WEIGHT),
        ('HSW Mut Threshold',   HSW_MUTABILITY_THRESHOLD),
        ('HSW Coev Pairs',      len(hsw_coev_pairs)),
        ('Hotspot Positions',   len(hotspot_positions)),
        ('Restricted Positions',len(restricted_positions)),
        ('Frozen Positions',    len(frozen_positions)),
        ('Max Hamming Cap',     MAX_HAMMING),
        ('Generations Run',     len(history)),
        ('Unique Sequences',    len(unique_inds)),
        ('Best Mutations',      describe_mutations(wt_seq, top_inds[0].seq)),
        ('Best Pred pH',        round(top_inds[0].fitness if goal=='maximize' else -top_inds[0].fitness, 4)),
        ('Best ΔpH vs WT',      round((top_inds[0].fitness if goal=='maximize' else -top_inds[0].fitness)-wt_pred, 4)),
        ('Best Hamming Dist',   hamming(wt_seq, top_inds[0].seq)),
    ]
    _hdr(ws1,1,1,'Parameter'); _hdr(ws1,1,2,'Value')
    for ri,(k,v) in enumerate(summary_rows,2):
        bg='FFF5F5F5' if ri%2==0 else 'FFFFFFFF'
        _cell(ws1,ri,1,k,bg=bg,align='left'); _cell(ws1,ri,2,v,bg=bg,align='left')
    ws1.freeze_panes='A2'; _aw(ws1)

    # ── Sheet 2: Top 50 sequences ────────────────────────────────
    ws2 = wb.create_sheet('Top50_Sequences')
    for ci,h in enumerate(['Rank','Pred_pH','ΔpH_vs_WT','N_Mutations',
                           'Hamming_Dist','Similarity_%','Mutations','Sequence'],1):
        _hdr(ws2,1,ci,h)
    for ri,ind in enumerate(top_inds[:50],2):
        ph  = ind.fitness if goal=='maximize' else -ind.fitness
        d   = ph-wt_pred
        mts = describe_mutations(wt_seq,ind.seq)
        nm  = sum(1 for w,m in zip(wt_seq,ind.seq) if w!=m)
        hd  = hamming(wt_seq, ind.seq)
        sim = round(hamming_similarity(wt_seq, ind.seq)*100, 1)
        max_h = MAX_HAMMING if MAX_HAMMING else len(wt_seq)
        hd_frac = hd / max(max_h, 1)
        hd_bg = ('FFCCFFCC' if hd_frac<=0.3 else 'FFFFFF99' if hd_frac<=0.7 else 'FFFFCCCC')
        bg  = rank_bg(ri-1)
        _cell(ws2,ri,1,ri-1,       bg=bg,      fmt='0')
        _cell(ws2,ri,2,round(ph,4),bg=ph_bg(ph),fmt='0.0000')
        _cell(ws2,ri,3,round(d,4), bg=bg,      fmt='+0.0000;-0.0000;0.0000')
        _cell(ws2,ri,4,nm,         bg=bg,      fmt='0')
        _cell(ws2,ri,5,hd,         bg=hd_bg,    fmt='0')
        _cell(ws2,ri,6,sim,        bg=hd_bg,    fmt='0.0')
        _cell(ws2,ri,7,mts,        bg=bg,      align='left')
        _cell(ws2,ri,8,ind.seq,    bg=bg,      align='left')
    ws2.freeze_panes='A2'; _aw(ws2)

    # ── Sheet 3: Generation history ───────────────────────────────
    ws3 = wb.create_sheet('Generation_History')
    hcols = ['Generation','Best_pH','Mean_pH','Std_pH','ΔBest_vs_WT',
             'Temperature','Mutation_Rate','Phe_Weight','Tournament_K',
             'N_Banked','N_Tabu','Hamming_Best','Hamming_Mean','Hamming_Max','Best_Mutations']
    for ci,h in enumerate(hcols,1): _hdr(ws3,1,ci,h)
    for ri,h in enumerate(history,2):
        d  = h['best_pH']-wt_pred
        bg = 'FFF5F5F5' if ri%2==0 else 'FFFFFFFF'
        vals = [h['generation'],round(h['best_pH'],4),round(h['mean_pH'],4),
                round(h['std_fitness'],4),round(d,4),
                round(h['temperature'],4),round(h['mutation_rate'],4),
                round(h['phe_weight'],4),h['tournament_k'],
                h['n_banked'],h['n_tabu_pos'],
                h['hamming_best'],h['hamming_mean'],h['hamming_max'],
                h['best_mutations']]
        fmts = ['0','0.0000','0.0000','0.0000','+0.0000;-0.0000;0.0000',
                '0.0000','0.0000','0.0000','0','0','0',
                '0','0.00','0','@']
        for ci,(v,f) in enumerate(zip(vals,fmts),1):
            xbg = ph_bg(h['best_pH']) if ci==2 else bg
            _cell(ws3,ri,ci,v,bg=xbg,fmt=f)
    ws3.freeze_panes='A2'; _aw(ws3)

    # ── Sheet 4: Banked mutations ─────────────────────────────────
    ws4 = wb.create_sheet('Banked_Mutations')
    for ci,h in enumerate(['Mutation','Position','WT_AA','Mut_AA','Best_ΔpH'],1):
        _hdr(ws4,1,ci,h)
    for ri,((pos,waa,maa),delta) in enumerate(sorted(memory.banked.items(),key=lambda x:-x[1]),2):
        bg = 'FFF5F5F5' if ri%2==0 else 'FFFFFFFF'
        _cell(ws4,ri,1,f'{waa}{pos+1}{maa}',bg=bg)
        _cell(ws4,ri,2,pos+1,bg=bg,fmt='0')
        _cell(ws4,ri,3,waa,  bg=bg)
        _cell(ws4,ri,4,maa,  bg=bg)
        _cell(ws4,ri,5,round(delta,4),bg='FFCCE5CC',fmt='+0.0000;-0.0000;0.0000')
    ws4.freeze_panes='A2'; _aw(ws4)

    # ── Sheet 5: Mutation frequency in top 50 ─────────────────────
    ws5 = wb.create_sheet('Mutation_Frequency')
    for ci,h in enumerate(['Rank','Mutation','Count','Frequency_%'],1):
        _hdr(ws5,1,ci,h)
    ctr = Counter()
    for ind in top_inds[:50]:
        for i,(w,m) in enumerate(zip(wt_seq,ind.seq)):
            if w!=m: ctr[f'{w}{i+1}{m}']+=1
    for ri,(mut,cnt) in enumerate(ctr.most_common(50),2):
        bg = 'FFF5F5F5' if ri%2==0 else 'FFFFFFFF'
        _cell(ws5,ri,1,ri-1,                bg=bg,fmt='0')
        _cell(ws5,ri,2,mut,                 bg=bg)
        _cell(ws5,ri,3,cnt,                 bg=bg,fmt='0')
        _cell(ws5,ri,4,round(cnt/50*100,1), bg=bg,fmt='0.0')
    ws5.freeze_panes='A2'; _aw(ws5)

    # ── Sheet 6: HSW-guided hits ──────────────────────────────────
    ws6 = wb.create_sheet('HSW_Position_Hits')
    hsw_positions = set()
    for p1,p2,_ in hsw_coev_pairs:
        hsw_positions.add(p1+1); hsw_positions.add(p2+1)
    for ci,h in enumerate(['Rank','Mutation','Position_in_HSW','ΔpH','Sequence'],1):
        _hdr(ws6,1,ci,h)
    ri = 2
    for rank,ind in enumerate(top_inds[:50],1):
        ph = ind.fitness if goal=='maximize' else -ind.fitness
        for i,(w,m) in enumerate(zip(wt_seq,ind.seq)):
            if w!=m and (i+1) in hsw_positions:
                bg = 'FFFFE0B2'
                _cell(ws6,ri,1,rank,           bg=bg,fmt='0')
                _cell(ws6,ri,2,f'{w}{i+1}{m}', bg=bg)
                _cell(ws6,ri,3,i+1,            bg=bg,fmt='0')
                _cell(ws6,ri,4,round(ph-wt_pred,4),bg=ph_bg(ph),fmt='+0.0000;-0.0000;0.0000')
                _cell(ws6,ri,5,ind.seq,        bg='FFFFFFFF',align='left')
                ri+=1
    ws6.freeze_panes='A2'; _aw(ws6)

    wb.save('aco_ga_results.xlsx')
    print('✓ Saved: aco_ga_results.xlsx')


write_excel(unique_inds, history, wt_seq, wt_pred, memory, GA_GOAL)

✓ Saved: aco_ga_results.xlsx


---
## 19 · Final summary

In [49]:
print('='*65)
print('  ACO-GA — FINAL SUMMARY')
print('='*65)
print(f'  WT header            : {wt_header}')
print(f'  WT length            : {len(wt_seq)} aa')
print(f'  Predicted WT pH      : {wt_pred:.4f}')
print(f'  Goal                 : {GA_GOAL}')
print(f'  Model                : {MODEL_NAME}  (Bootstrap augmentation)')
print(f'  Model Test R²        : {metadata["test_r2"]:.4f}')
print(f'  PSSM weight          : {PSSM_WEIGHT}')
print(f'  HSW hotspot positions: {len(hotspot_positions)}')
print(f'  HSW restricted pos   : {len(restricted_positions)}')
print(f'  HSW frozen pos       : {len(frozen_positions)}')
print(f'  HSW coev pairs       : {len(hsw_coev_pairs)}')
print(f'  Generations run      : {len(history)}')
print(f'  Unique sequences     : {len(unique_inds)}')
print(f'  MAX_HAMMING cap      : {MAX_HAMMING}')
print()
print(f'  🏆  BEST RESULT')
print(f'      Mutations   : {describe_mutations(wt_seq, unique_inds[0].seq)}')
print(f'      Pred pH     : {(unique_inds[0].fitness if GA_GOAL=="maximize" else -unique_inds[0].fitness):.4f}')
print(f'      ΔpH vs WT   : {(unique_inds[0].fitness if GA_GOAL=="maximize" else -unique_inds[0].fitness) - wt_pred:+.4f}')
print(f'      Hamming dist: {hamming(wt_seq, unique_inds[0].seq)}  '
      f'(similarity {hamming_similarity(wt_seq, unique_inds[0].seq)*100:.1f}%)')
print()
print('  Output files:')
print('    📊 aco_ga_results.xlsx    — 6 sheets: Summary / Top50 / History / Banked / Freq / HSW-hits')
print('    📈 aco_ga_convergence.png — pH + schedules + memory + Hamming panels')
print('='*65)

  ACO-GA — FINAL SUMMARY
  WT header            : Afecal Nit
  WT length            : 356 aa
  Predicted WT pH      : 7.2550
  Goal                 : maximize
  Model                : SVR_Linear  (Bootstrap augmentation)
  Model Test R²        : 0.7068
  PSSM weight          : 0.5
  HSW hotspot positions: 142
  HSW restricted pos   : 214
  HSW frozen pos       : 0
  HSW coev pairs       : 10209
  Generations run      : 80
  Unique sequences     : 11086
  MAX_HAMMING cap      : 10

  🏆  BEST RESULT
      Mutations   : L31M, V44Q, S69K, F84E, A91F, D118I, T138P, V139Q, H198C, L262T
      Pred pH     : 7.3201
      ΔpH vs WT   : +0.0651
      Hamming dist: 10  (similarity 97.2%)

  Output files:
    📊 aco_ga_results.xlsx    — 6 sheets: Summary / Top50 / History / Banked / Freq / HSW-hits
    📈 aco_ga_convergence.png — pH + schedules + memory + Hamming panels
